<a href="https://www.kaggle.com/code/nityaverma19/item-based-cf?scriptVersionId=167532473" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/netflix-prize-data/combined_data_3.txt
/kaggle/input/netflix-prize-data/movie_titles.csv
/kaggle/input/netflix-prize-data/combined_data_4.txt
/kaggle/input/netflix-prize-data/combined_data_1.txt
/kaggle/input/netflix-prize-data/README
/kaggle/input/netflix-prize-data/probe.txt
/kaggle/input/netflix-prize-data/combined_data_2.txt
/kaggle/input/netflix-prize-data/qualifying.txt


In [2]:
# Defining column names
column_names = ['MovieID', 'YearOfRelease', 'Title']

# Reading CSV file without headers, skipping lines with parsing errors
movie_titles = pd.read_csv('/kaggle/input/netflix-prize-data/movie_titles.csv', encoding='ISO-8859-1', header=None, names=column_names,on_bad_lines ='skip')

# Display the first few rows of the dataframe
movie_titles.head(10)

,MovieID,YearOfRelease,Title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW
5,6,1997.0,Sick
6,7,1992.0,8 Man
7,8,2004.0,What the #$*! Do We Know!?
8,9,1991.0,Class of Nuke 'Em High 2
9,10,2001.0,Fighter


Txt to df -  combined data 1

In [3]:
import pandas as pd

def read_combined_data(file_path):
    # Initialize lists to store data
    movie_ids = []
    customer_ids = []
    ratings = []
    dates = []

    # Read the file
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Initialize variables
    current_movie_id = None

    # Process each line
    for line in lines:
        # Strip leading/trailing whitespaces
        line = line.strip()

        # Check for movie ID
        if line.endswith(':'):
            # Update current movie ID
            current_movie_id = line[:-1].strip()
        elif line:  # Check if the line is not empty
            # Extract customer data
            data = line.strip().split(',')
            if len(data) >= 3:
                customer_id = data[0].strip()
                rating_str = data[1].strip()
                date = data[2].strip()

                try:
                    rating = float(rating_str)
                except ValueError:
                    print(f"Skipping invalid rating (not a valid float): '{rating_str}'")
                    continue

                # Append data to lists
                movie_ids.append(current_movie_id)
                customer_ids.append(customer_id)
                ratings.append(rating)
                dates.append(date)
            else:
                print(f"Skipping line with insufficient data: '{line}'")

    # Create DataFrame
    combined_data = pd.DataFrame({
        'MovieID': movie_ids,
        'CustomerID': customer_ids,
        'Rating': ratings,
        'Date': dates
    })

    return combined_data

# Read combined_data_1.txt
file_path = "/kaggle/input/netflix-prize-data/combined_data_1.txt"
print(f"Reading file: {file_path}")
data_1 = read_combined_data(file_path)

# Display the first few rows of the DataFrame
print(data_1.head())

Reading file: /kaggle/input/netflix-prize-data/combined_data_1.txt
  MovieID CustomerID  Rating        Date
0       1    1488844     3.0  2005-09-06
1       1     822109     5.0  2005-05-13
2       1     885013     4.0  2005-10-19
3       1      30878     4.0  2005-12-26
4       1     823519     3.0  2004-05-03


In [4]:
data_1.head(5)

,MovieID,CustomerID,Rating,Date
0,1,1488844,3.0,2005-09-06
1,1,822109,5.0,2005-05-13
2,1,885013,4.0,2005-10-19
3,1,30878,4.0,2005-12-26
4,1,823519,3.0,2004-05-03


txt to df- combined data 2



In [5]:
data_1.drop(columns = "Date", inplace = True)

In [6]:
data_1.head()

,MovieID,CustomerID,Rating
0,1,1488844,3.0
1,1,822109,5.0
2,1,885013,4.0
3,1,30878,4.0
4,1,823519,3.0


**Viewing the Data**

*Total movies*

In [7]:
movie_titles.shape

(17434, 3)

Total movies - 17434

*Total ratings in combined_data_1*

In [8]:
data_1.shape

(24053764, 3)

*Total movies in data_1*

In [9]:
data_1['MovieID'].tail(1)

24053763    4499
Name: MovieID, dtype: object

4499 movies rated in data_1

*Total ratings for movie*

In [10]:
data_1[data_1['MovieID'] == '1'].shape

(547, 3)

547 ratings for movie 1

***Changing the data type of rating***

In [11]:
data_1['Rating']=data_1['Rating'].astype("int8")

***Unique users and movies***

In [12]:
data_1['CustomerID'].nunique()

470758

In [13]:
data_1['MovieID'].nunique()

4499

# **------------------------------------------------------------------------------------------------------------------------------------------------**

## **DATA CLEANING**

### **Missing Values**

In [14]:
movie_titles.isnull().sum()

MovieID          0
YearOfRelease    7
Title            0
dtype: int64

*no missing values in movie titles dataset*

***Dropping year of release***

In [15]:
movie_titles.drop(columns = "YearOfRelease", inplace = True)

In [16]:
data_1.isnull().sum()

MovieID       0
CustomerID    0
Rating        0
dtype: int64

*No missing values in the data_1*

### **Duplicates**

***Movie titles dataset*** 

In [17]:
movie_titles.duplicated().sum()

0

*No duplicated movies found*

In [18]:
data_1.head()

,MovieID,CustomerID,Rating
0,1,1488844,3
1,1,822109,5
2,1,885013,4
3,1,30878,4
4,1,823519,3


***Any user who has rated the same movie more than once***

In [19]:
data_1.groupby(['MovieID', 'CustomerID']).count()

Rating
MovieID CustomerID        
1       1001779          1
        1005769          1
        1008986          1
        1009622          1
        1011918          1
...                    ...
999     984474           1
        98676            1
        995497           1
        997271           1
        999528           1

[24053764 rows x 1 columns]

In [20]:
data_1.groupby(['MovieID', 'CustomerID']).count()['Rating'].sum()

24053764

*No one user has rated the movie movie more than once*

***Maximum and minimum values of rating***

In [21]:
print("max",data_1['Rating'].max())
print("min",data_1['Rating'].min())

max 5
min 1


## *Merging movie titles with ratings data*

### ***Changing the data type of MovieID***

In [22]:
movie_titles['MovieID'] = movie_titles['MovieID'].astype('int16')

In [23]:
data_1['MovieID'] = data_1['MovieID'].astype('int16')

In [24]:
data1_titles = data_1.merge(movie_titles, on = 'MovieID', how = 'inner')

### **-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------**

# *Collaborative Filtering*

## **1) Item Based Collaborative Filtering:**

### Makes recommendations based on item-item interaction. 


In [25]:
data1_titles.groupby('CustomerID').count().sort_values(by = 'MovieID', ascending = False)

,MovieID,Rating,Title
CustomerID,,,
305344,4382,4382,4382
387418,4337,4337,4337
2439493,4115,4115,4115
1664010,3940,3940,3940
2118461,3695,3695,3695
...,...,...,...
2312678,1,1,1
2557114,1,1,1
853003,1,1,1


### *User criteria- rated more than 100 movies*

In [26]:
rating_by_each_user = data1_titles.groupby('CustomerID').count()['MovieID']>100

In [27]:
#all users who have rated more than 100 movies
active_users = rating_by_each_user[rating_by_each_user].index
active_users = active_users.tolist()
active_users

['1000062',
 '1000079',
 '1000084',
 '1000095',
 '1000192',
 '1000301',
 '1000328',
 '1000380',
 '1000387',
 '1000406',
 '1000410',
 '1000527',
 '1000530',
 '1000554',
 '1000569',
 '100057',
 '1000596',
 '1000634',
 '10007',
 '1000710',
 '1000774',
 '1000779',
 '1000840',
 '1000851',
 '1000880',
 '1000904',
 '100093',
 '1000965',
 '1000976',
 '1001004',
 '1001046',
 '1001129',
 '1001140',
 '1001163',
 '1001167',
 '1001185',
 '100121',
 '1001299',
 '1001371',
 '1001375',
 '1001376',
 '1001404',
 '1001454',
 '1001461',
 '1001464',
 '1001477',
 '10016',
 '1001653',
 '1001654',
 '1001660',
 '1001701',
 '1001779',
 '1001788',
 '1001797',
 '1001833',
 '1001878',
 '1001912',
 '1001926',
 '1001928',
 '100197',
 '1002025',
 '1002117',
 '100213',
 '1002135',
 '1002150',
 '1002208',
 '1002249',
 '1002256',
 '1002277',
 '100228',
 '1002329',
 '1002367',
 '1002407',
 '1002412',
 '1002418',
 '1002426',
 '1002453',
 '1002491',
 '1002501',
 '1002504',
 '1002509',
 '1002524',
 '1002575',
 '1002587',
 '

*Total 70270 users have rated more than 100 movies*

In [28]:
filtered_data = data1_titles[data1_titles['CustomerID'].isin(active_users)]
filtered_data

,MovieID,CustomerID,Rating,Title
0,1,1488844,3,Dinosaur Planet
3,1,30878,4,Dinosaur Planet
4,1,823519,3,Dinosaur Planet
5,1,893988,3,Dinosaur Planet
7,1,1248029,3,Dinosaur Planet
...,...,...,...,...
23807397,4499,2219917,3,In My Skin
23807398,4499,1796454,1,In My Skin
23807400,4499,2591364,2,In My Skin
23807403,4499,988963,3,In My Skin


### *Movie criteria- considering movies which have more than 100 ratings*

In [29]:
most_rated_movies = filtered_data.groupby('MovieID').count()['Rating']>100

In [30]:
y = most_rated_movies[most_rated_movies].index
y = y.tolist()

In [31]:
final_data_1 = filtered_data[filtered_data['MovieID'].isin(y)]
final_data_1

,MovieID,CustomerID,Rating,Title
0,1,1488844,3,Dinosaur Planet
3,1,30878,4,Dinosaur Planet
4,1,823519,3,Dinosaur Planet
5,1,893988,3,Dinosaur Planet
7,1,1248029,3,Dinosaur Planet
...,...,...,...,...
23807397,4499,2219917,3,In My Skin
23807398,4499,1796454,1,In My Skin
23807400,4499,2591364,2,In My Skin
23807403,4499,988963,3,In My Skin


### ***Making a user-movie matrix***

In [32]:
user_movie_interact = final_data_1.pivot_table(index = 'Title', columns = 'CustomerID', values = 'Rating')
user_movie_interact.fillna(0, inplace = True)   #filling nan values with 0
user_movie_interact

CustomerID,1000062,1000079,1000084,1000095,1000192,1000301,1000328,1000380,1000387,1000406,...,999598,999601,999663,999693,999756,999768,999836,999901,99993,999944
Title,,,,,,,,,,,,,,,,,,,,,
'N Sync: 'N the Mix,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
... And God Spoke,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,0.0,3.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10 Things I Hate About You,0.0,3.0,5.0,0.0,2.0,0.0,0.0,5.0,5.0,0.0,...,0.0,0.0,0.0,2.0,5.0,0.0,2.0,0.0,4.0,4.0
101 Dalmatians II: Patch's London Adventure,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zhou Yu's Train,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Ziegfeld Girl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zig Zag: Special Edition,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [33]:
user_movie_interact.dtypes

CustomerID
1000062    float64
1000079    float64
1000084    float64
1000095    float64
1000192    float64
            ...   
999768     float64
999836     float64
999901     float64
99993      float64
999944     float64
Length: 69334, dtype: object

In [34]:
#reducing the size of the data

user_movie_interact = user_movie_interact.astype('int8')

### ***Finding similarity between users***

### *a) Cosine Similarity*

*Cosine similarity is unaffected by the magnitude and focuses on the direction, hence it is beneficial to use cosine similarity here as we have sparse data with many 0*

![](https://builtin.com/sites/www.builtin.com/files/styles/ckeditor_optimize/public/inline-images/1_cosine-similarity.png)

In [35]:
from sklearn.metrics.pairwise import cosine_similarity

In [36]:
similarity_score = cosine_similarity(user_movie_interact)
similarity_score

array([[1.        , 0.01987857, 0.03641458, ..., 0.03368261, 0.01767937,
        0.01110233],
       [0.01987857, 1.        , 0.02624803, ..., 0.0254308 , 0.03517933,
        0.02405177],
       [0.03641458, 0.02624803, 1.        , ..., 0.03328357, 0.01921015,
        0.0194061 ],
       ...,
       [0.03368261, 0.0254308 , 0.03328357, ..., 1.        , 0.03761942,
        0.03117729],
       [0.01767937, 0.03517933, 0.01921015, ..., 0.03761942, 1.        ,
        0.02759687],
       [0.01110233, 0.02405177, 0.0194061 , ..., 0.03117729, 0.02759687,
        1.        ]])

In [37]:
similarity_score.shape

(3363, 3363)

In [38]:
#converting to a dataframe
similar_movie_df = pd.DataFrame(similarity_score, index = user_movie_interact.index, columns = user_movie_interact.index)
similar_movie_df

Title,'N Sync: 'N the Mix,... And God Spoke,10,10 Things I Hate About You,101 Dalmatians II: Patch's London Adventure,11:14,13 Ghosts,13 Moons,18 Again,1942: A Love Story,...,Yu-Gi-Oh!: The Movie,Zatoichi,Zatoichi the Outlaw,Zatoichi's Conspiracy,Zeus and Roxanne,Zhou Yu's Train,Ziegfeld Girl,Zig Zag: Special Edition,Zombie 3,s-Cry-ed
Title,,,,,,,,,,,,,,,,,,,,,
'N Sync: 'N the Mix,1.000000,0.019879,0.036415,0.076004,0.052879,0.008988,0.042770,0.013892,0.045510,0.022889,...,0.017440,0.008932,0.013266,0.014391,0.052914,0.013267,0.014118,0.033683,0.017679,0.011102
... And God Spoke,0.019879,1.000000,0.026248,0.019484,0.012649,0.010644,0.016177,0.017197,0.029949,0.038503,...,0.025311,0.027359,0.031167,0.033848,0.017831,0.011229,0.041705,0.025431,0.035179,0.024052
10,0.036415,0.026248,1.000000,0.149125,0.115047,0.044687,0.092642,0.030907,0.135054,0.027134,...,0.036466,0.026710,0.022148,0.018977,0.085168,0.027154,0.054368,0.033284,0.019210,0.019406
10 Things I Hate About You,0.076004,0.019484,0.149125,1.000000,0.218626,0.072864,0.230402,0.028362,0.158377,0.034361,...,0.087131,0.027298,0.024374,0.019442,0.124525,0.024146,0.034671,0.039165,0.015717,0.038140
101 Dalmatians II: Patch's London Adventure,0.052879,0.012649,0.115047,0.218626,1.000000,0.038743,0.154057,0.019101,0.121114,0.026652,...,0.119727,0.018514,0.018310,0.020956,0.192647,0.013384,0.034684,0.034627,0.012418,0.022152
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zhou Yu's Train,0.013267,0.011229,0.027154,0.024146,0.013384,0.026642,0.012089,0.019879,0.016585,0.040255,...,0.021884,0.035456,0.031640,0.036774,0.012687,1.000000,0.016726,0.013635,0.022087,0.018914
Ziegfeld Girl,0.014118,0.041705,0.054368,0.034671,0.034684,0.025955,0.032280,0.028268,0.035617,0.048304,...,0.020498,0.021417,0.037918,0.036422,0.023436,0.016726,1.000000,0.018079,0.038978,0.021145
Zig Zag: Special Edition,0.033683,0.025431,0.033284,0.039165,0.034627,0.047170,0.047750,0.039029,0.039351,0.026033,...,0.035091,0.018818,0.016937,0.025364,0.039336,0.013635,0.018079,1.000000,0.037619,0.031177


In [39]:
print(similar_movie_df.columns)


Index([''N Sync: 'N the Mix', '... And God Spoke', '10',
       '10 Things I Hate About You',
       '101 Dalmatians II: Patch's London Adventure', '11:14', '13 Ghosts',
       '13 Moons', '18 Again', '1942: A Love Story',
       ...
       'Yu-Gi-Oh!: The Movie', 'Zatoichi', 'Zatoichi the Outlaw',
       'Zatoichi's Conspiracy', 'Zeus and Roxanne', 'Zhou Yu's Train',
       'Ziegfeld Girl', 'Zig Zag: Special Edition', 'Zombie 3', 's-Cry-ed'],
      dtype='object', name='Title', length=3363)


In [40]:
#if a user has rated movie 1 a 3 then in that case we will scale it, i.e all the values will be multiplied by 3
score = similar_movie_df['Zombie 3']*5
score

Title
'N Sync: 'N the Mix                            0.088397
... And God Spoke                              0.175897
10                                             0.096051
10 Things I Hate About You                     0.078585
101 Dalmatians II: Patch's London Adventure    0.062090
                                                 ...   
Zhou Yu's Train                                0.110433
Ziegfeld Girl                                  0.194890
Zig Zag: Special Edition                       0.188097
Zombie 3                                       5.000000
s-Cry-ed                                       0.137984
Name: Zombie 3, Length: 3363, dtype: float64

In [41]:
#then we will arrange it in descending order to get the most similar movie

score.sort_values(ascending = False)

Title
Zombie 3                                                5.000000
Lucio Fulci: The Beyond                                 1.325837
Dario Argento Collection: Vol. 2: Demons 2              1.322847
Lucio Fulci: The New York Ripper                        1.305854
The Crazies                                             1.255410
                                                          ...   
Saving Private Ryan: Bonus Material                     0.007821
Minority Report: Bonus Material                         0.006412
Capturing the Friedmans: Bonus Material                 0.005463
The Terminal: Bonus Material                            0.000000
Battlestar Galactica: The Miniseries: Bonus Material    0.000000
Name: Zombie 3, Length: 3363, dtype: float64

*We will ignore the first value*
*Movie 1 is the most similar to movie 694*

In [42]:
#if the user has rated a movie 1 i.e the user doesn't like the movie
score = similar_movie_df['Zombie 3']*1
score

Title
'N Sync: 'N the Mix                            0.017679
... And God Spoke                              0.035179
10                                             0.019210
10 Things I Hate About You                     0.015717
101 Dalmatians II: Patch's London Adventure    0.012418
                                                 ...   
Zhou Yu's Train                                0.022087
Ziegfeld Girl                                  0.038978
Zig Zag: Special Edition                       0.037619
Zombie 3                                       1.000000
s-Cry-ed                                       0.027597
Name: Zombie 3, Length: 3363, dtype: float64

In [43]:
score.sort_values(ascending = False)

Title
Zombie 3                                                1.000000
Lucio Fulci: The Beyond                                 0.265167
Dario Argento Collection: Vol. 2: Demons 2              0.264569
Lucio Fulci: The New York Ripper                        0.261171
The Crazies                                             0.251082
                                                          ...   
Saving Private Ryan: Bonus Material                     0.001564
Minority Report: Bonus Material                         0.001282
Capturing the Friedmans: Bonus Material                 0.001093
The Terminal: Bonus Material                            0.000000
Battlestar Galactica: The Miniseries: Bonus Material    0.000000
Name: Zombie 3, Length: 3363, dtype: float64

*In this case we are still getting the similar movies but in reality the user doesn't like the movie so we don't want to recommend it, hence we will subtract the rating by the mean i.e 2.5*

In [44]:
score = similar_movie_df['Zombie 3']*(1-2.5)
score

Title
'N Sync: 'N the Mix                           -0.026519
... And God Spoke                             -0.052769
10                                            -0.028815
10 Things I Hate About You                    -0.023576
101 Dalmatians II: Patch's London Adventure   -0.018627
                                                 ...   
Zhou Yu's Train                               -0.033130
Ziegfeld Girl                                 -0.058467
Zig Zag: Special Edition                      -0.056429
Zombie 3                                      -1.500000
s-Cry-ed                                      -0.041395
Name: Zombie 3, Length: 3363, dtype: float64

In [45]:
score.sort_values(ascending = False)

Title
The Terminal: Bonus Material                           -0.000000
Battlestar Galactica: The Miniseries: Bonus Material   -0.000000
Capturing the Friedmans: Bonus Material                -0.001639
Minority Report: Bonus Material                        -0.001924
Saving Private Ryan: Bonus Material                    -0.002346
                                                          ...   
The Crazies                                            -0.376623
Lucio Fulci: The New York Ripper                       -0.391756
Dario Argento Collection: Vol. 2: Demons 2             -0.396854
Lucio Fulci: The Beyond                                -0.397751
Zombie 3                                               -1.500000
Name: Zombie 3, Length: 3363, dtype: float64

*Now only if the rating is postive i.e 3,4,5 only then it will keep the movie on the top of the list*

### *Making a function to recommend movies*

In [46]:
def recommend(movie_id, rating):
    score = similar_movie_df[movie_id]*(rating-2.5)
    score = score.sort_values(ascending = False)[1:6]    #recommend top 5 movies
    
    return score


In [47]:
recommend('1942: A Love Story',5)

Title
Hum Aapke Hain Koun    1.376405
Andaz Apna Apna        1.350060
Baazigar               1.308477
Saajan                 1.297157
Satte Pe Satta         1.259409
Name: 1942: A Love Story, dtype: float64

ISSUE: 
* Cold start problem
* Live data